# Logistic Regression: Complete Guide

## Learning Objectives
By the end of this notebook, you will:
- Understand logistic regression for classification
- Implement logistic regression from scratch
- Master the sigmoid function and cross-entropy loss
- Handle binary and multiclass classification
- Interpret coefficients as odds ratios
- Evaluate classification models with proper metrics

## Prerequisites
- Linear regression understanding
- Basic probability concepts
- Calculus fundamentals

---
## Part 1: From Linear to Logistic

### The Problem with Linear Regression for Classification
Linear regression outputs can be any real number, but classification needs probabilities (0 to 1).

### The Solution: Sigmoid Function
The sigmoid function squashes any input to the range (0, 1):

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

### The Logistic Regression Model
$$P(y=1|x) = \sigma(\beta_0 + \beta_1 x_1 + ... + \beta_p x_p)$$

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer, load_iris, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report,
                             roc_curve, auc, precision_recall_curve)
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
print("Libraries loaded successfully!")

In [ ]:
# Visualize the sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-10, 10, 100)

plt.figure(figsize=(10, 6))
plt.plot(z, sigmoid(z), 'b-', lw=3, label='Sigmoid: σ(z)')
plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.7, label='Decision boundary (0.5)')
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
plt.fill_between(z, sigmoid(z), 0.5, where=(sigmoid(z) > 0.5), alpha=0.3, color='green', label='Predict Class 1')
plt.fill_between(z, sigmoid(z), 0.5, where=(sigmoid(z) <= 0.5), alpha=0.3, color='red', label='Predict Class 0')
plt.xlabel('z = βx', fontsize=12)
plt.ylabel('σ(z) = P(y=1|x)', fontsize=12)
plt.title('The Sigmoid Function', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Key properties:")
print("  • σ(0) = 0.5")
print("  • As z → +∞, σ(z) → 1")
print("  • As z → -∞, σ(z) → 0")

---
## Part 2: Logistic Regression from Scratch

### Binary Cross-Entropy Loss
$$L = -\frac{1}{n} \sum_{i=1}^{n} [y_i \log(\hat{y}_i) + (1-y_i) \log(1-\hat{y}_i)]$$

### Gradient
$$\nabla L = \frac{1}{n} X^T (\hat{y} - y)$$

Note: Same form as linear regression gradient!

In [ ]:
class LogisticRegressionScratch:
    """Logistic Regression implemented from scratch."""
    
    def __init__(self, learning_rate=0.01, n_iterations=1000, threshold=0.5):
        self.lr = learning_rate
        self.n_iter = n_iterations
        self.threshold = threshold
        self.weights = None
        self.bias = None
        self.loss_history = []
        
    def sigmoid(self, z):
        # Clip to prevent overflow
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        
        # Initialize weights
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        for i in range(self.n_iter):
            # Forward pass
            z = X @ self.weights + self.bias
            y_pred = self.sigmoid(z)
            
            # Compute loss (binary cross-entropy)
            epsilon = 1e-15  # Prevent log(0)
            loss = -np.mean(y * np.log(y_pred + epsilon) + 
                           (1 - y) * np.log(1 - y_pred + epsilon))
            self.loss_history.append(loss)
            
            # Compute gradients
            dw = (1 / n_samples) * X.T @ (y_pred - y)
            db = (1 / n_samples) * np.sum(y_pred - y)
            
            # Update weights
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
        return self
    
    def predict_proba(self, X):
        z = X @ self.weights + self.bias
        return self.sigmoid(z)
    
    def predict(self, X):
        return (self.predict_proba(X) >= self.threshold).astype(int)
    
    def score(self, X, y):
        return np.mean(self.predict(X) == y)

print("LogisticRegressionScratch class defined!")

In [ ]:
# Generate binary classification data
np.random.seed(42)
X_binary, y_binary = make_classification(
    n_samples=500, n_features=2, n_redundant=0,
    n_informative=2, n_clusters_per_class=1,
    class_sep=1.5, random_state=42
)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_binary, y_binary, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train our model
model_scratch = LogisticRegressionScratch(learning_rate=0.1, n_iterations=500)
model_scratch.fit(X_train_scaled, y_train)

print("From-Scratch Model Results:")
print(f"  Training Accuracy: {model_scratch.score(X_train_scaled, y_train):.2%}")
print(f"  Test Accuracy: {model_scratch.score(X_test_scaled, y_test):.2%}")

In [ ]:
# Visualize decision boundary
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Decision boundary
xx, yy = np.meshgrid(np.linspace(-3, 3, 100), np.linspace(-3, 3, 100))
Z = model_scratch.predict_proba(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

axes[0].contourf(xx, yy, Z, levels=50, cmap='RdYlBu', alpha=0.8)
axes[0].scatter(X_train_scaled[y_train==0, 0], X_train_scaled[y_train==0, 1], 
                c='red', edgecolors='black', label='Class 0', s=50)
axes[0].scatter(X_train_scaled[y_train==1, 0], X_train_scaled[y_train==1, 1], 
                c='blue', edgecolors='black', label='Class 1', s=50)
axes[0].contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].set_title('Decision Boundary')
axes[0].legend()

# Plot 2: Loss convergence
axes[1].plot(model_scratch.loss_history)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Binary Cross-Entropy Loss')
axes[1].set_title('Training Convergence')

plt.tight_layout()
plt.show()

---
## Part 3: Using Scikit-Learn

In [ ]:
# Compare with sklearn
from sklearn.linear_model import LogisticRegression

sklearn_model = LogisticRegression(max_iter=500)
sklearn_model.fit(X_train_scaled, y_train)

print("Comparison: Scratch vs Sklearn")
print("=" * 50)
print(f"{'Metric':<20} {'Scratch':<15} {'Sklearn':<15}")
print("-" * 50)
print(f"{'Train Accuracy':<20} {model_scratch.score(X_train_scaled, y_train):<15.4f} {sklearn_model.score(X_train_scaled, y_train):<15.4f}")
print(f"{'Test Accuracy':<20} {model_scratch.score(X_test_scaled, y_test):<15.4f} {sklearn_model.score(X_test_scaled, y_test):<15.4f}")

---
## Part 4: Classification Metrics

### Confusion Matrix
|  | Predicted Positive | Predicted Negative |
|--|-------------------|-------------------|
| **Actual Positive** | True Positive (TP) | False Negative (FN) |
| **Actual Negative** | False Positive (FP) | True Negative (TN) |

### Key Metrics
- **Accuracy**: $(TP + TN) / (TP + TN + FP + FN)$
- **Precision**: $TP / (TP + FP)$ - "Of predicted positives, how many correct?"
- **Recall**: $TP / (TP + FN)$ - "Of actual positives, how many found?"
- **F1 Score**: $2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}$

In [ ]:
# Breast cancer dataset
cancer = load_breast_cancer()
X_cancer = cancer.data
y_cancer = cancer.target

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cancer, y_cancer, test_size=0.2, random_state=42, stratify=y_cancer
)

scaler_c = StandardScaler()
X_train_c_scaled = scaler_c.fit_transform(X_train_c)
X_test_c_scaled = scaler_c.transform(X_test_c)

# Train model
model_cancer = LogisticRegression(max_iter=1000)
model_cancer.fit(X_train_c_scaled, y_train_c)
y_pred_c = model_cancer.predict(X_test_c_scaled)

print("Breast Cancer Classification Results:")
print(classification_report(y_test_c, y_pred_c, target_names=cancer.target_names))

In [ ]:
# Confusion matrix visualization
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Regular confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test_c, y_pred_c, display_labels=cancer.target_names, 
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('Confusion Matrix (Counts)')

# Normalized confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test_c, y_pred_c, display_labels=cancer.target_names, 
    cmap='Blues', normalize='true', ax=axes[1]
)
axes[1].set_title('Confusion Matrix (Normalized)')

plt.tight_layout()
plt.show()

---
## Part 5: ROC and Precision-Recall Curves

### ROC Curve
- X-axis: False Positive Rate (FPR) = FP / (FP + TN)
- Y-axis: True Positive Rate (TPR) = Recall = TP / (TP + FN)
- **AUC**: Area Under Curve (higher is better, 1.0 = perfect)

### Precision-Recall Curve
- Better for imbalanced datasets
- X-axis: Recall
- Y-axis: Precision

In [ ]:
# Get probability predictions
y_proba = model_cancer.predict_proba(X_test_c_scaled)[:, 1]

# ROC Curve
fpr, tpr, thresholds_roc = roc_curve(y_test_c, y_proba)
roc_auc = auc(fpr, tpr)

# Precision-Recall Curve
precision, recall, thresholds_pr = precision_recall_curve(y_test_c, y_proba)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
axes[0].plot(fpr, tpr, 'b-', lw=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'r--', label='Random Classifier')
axes[0].fill_between(fpr, tpr, alpha=0.3)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Precision-Recall Curve
axes[1].plot(recall, precision, 'g-', lw=2, label='PR Curve')
axes[1].fill_between(recall, precision, alpha=0.3, color='green')
axes[1].axhline(y=y_test_c.mean(), color='r', linestyle='--', label='Baseline')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"AUC-ROC: {roc_auc:.4f}")

In [ ]:
# Threshold selection
thresholds_to_try = [0.3, 0.5, 0.7, 0.9]

print("Effect of Different Thresholds:")
print("=" * 60)
print(f"{'Threshold':<12} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1':<12}")
print("-" * 60)

for thresh in thresholds_to_try:
    y_pred_thresh = (y_proba >= thresh).astype(int)
    acc = accuracy_score(y_test_c, y_pred_thresh)
    prec = precision_score(y_test_c, y_pred_thresh, zero_division=0)
    rec = recall_score(y_test_c, y_pred_thresh, zero_division=0)
    f1 = f1_score(y_test_c, y_pred_thresh, zero_division=0)
    print(f"{thresh:<12} {acc:<12.4f} {prec:<12.4f} {rec:<12.4f} {f1:<12.4f}")

---
## Part 6: Multiclass Classification

Logistic regression extends to multiple classes using:
- **One-vs-Rest (OvR)**: Train K binary classifiers
- **Softmax (Multinomial)**: Single model with softmax output

### Softmax Function
$$P(y=k|x) = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$$

In [ ]:
# Iris dataset (3 classes)
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42, stratify=y_iris
)

scaler_i = StandardScaler()
X_train_i_scaled = scaler_i.fit_transform(X_train_i)
X_test_i_scaled = scaler_i.transform(X_test_i)

# Compare OvR vs Multinomial
models_multi = {
    'One-vs-Rest': LogisticRegression(multi_class='ovr', max_iter=1000),
    'Multinomial': LogisticRegression(multi_class='multinomial', max_iter=1000)
}

for name, model in models_multi.items():
    model.fit(X_train_i_scaled, y_train_i)
    train_acc = model.score(X_train_i_scaled, y_train_i)
    test_acc = model.score(X_test_i_scaled, y_test_i)
    print(f"{name}: Train={train_acc:.2%}, Test={test_acc:.2%}")

In [ ]:
# Multiclass confusion matrix
model_multi = models_multi['Multinomial']
y_pred_i = model_multi.predict(X_test_i_scaled)

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test_i, y_pred_i, display_labels=iris.target_names,
    cmap='Blues', ax=ax
)
ax.set_title('Multiclass Confusion Matrix (Iris)')
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(y_test_i, y_pred_i, target_names=iris.target_names))

---
## Part 7: Regularization in Logistic Regression

Scikit-learn uses the `C` parameter (inverse of regularization strength):
- Smaller C → More regularization
- Larger C → Less regularization (closer to standard logistic regression)

In [ ]:
# Effect of regularization
C_values = [0.001, 0.01, 0.1, 1, 10, 100]

results_reg = []
for C in C_values:
    model = LogisticRegression(C=C, max_iter=1000)
    model.fit(X_train_c_scaled, y_train_c)
    results_reg.append({
        'C': C,
        'Train Acc': model.score(X_train_c_scaled, y_train_c),
        'Test Acc': model.score(X_test_c_scaled, y_test_c),
        'Non-zero coefs': np.sum(model.coef_ != 0)
    })

results_reg_df = pd.DataFrame(results_reg)
print("Effect of Regularization (C parameter):")
display(results_reg_df.round(4))

In [ ]:
# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogx(C_values, [r['Train Acc'] for r in results_reg], 'b-o', label='Train')
ax.semilogx(C_values, [r['Test Acc'] for r in results_reg], 'r-s', label='Test')
ax.set_xlabel('C (Inverse Regularization)')
ax.set_ylabel('Accuracy')
ax.set_title('Effect of Regularization Strength')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

---
## Part 8: Coefficient Interpretation

### Odds Ratio
The coefficient $\beta_j$ represents the log odds ratio:
$$e^{\beta_j} = \text{Odds Ratio}$$

- $e^{\beta_j} > 1$: Feature increases probability of positive class
- $e^{\beta_j} < 1$: Feature decreases probability of positive class
- $e^{\beta_j} = 1$: Feature has no effect

In [ ]:
# Coefficient interpretation for breast cancer
model_interp = LogisticRegression(max_iter=1000)
model_interp.fit(X_train_c_scaled, y_train_c)

coef_df = pd.DataFrame({
    'Feature': cancer.feature_names,
    'Coefficient': model_interp.coef_[0],
    'Odds Ratio': np.exp(model_interp.coef_[0])
}).sort_values('Coefficient', key=abs, ascending=False).head(10)

print("Top 10 Most Important Features (Breast Cancer):")
display(coef_df.round(4))

print("\nInterpretation:")
print("  Positive coefficient → Higher odds of MALIGNANT")
print("  Negative coefficient → Higher odds of BENIGN")

---
## Summary

### Key Takeaways

| Concept | Description |
|---------|-------------|
| **Sigmoid** | Maps any input to (0, 1) probability |
| **Cross-Entropy** | Loss function for classification |
| **Threshold** | Default 0.5, adjust for precision/recall tradeoff |
| **ROC-AUC** | Overall classifier quality (0.5 = random, 1.0 = perfect) |
| **Multiclass** | OvR or Softmax (multinomial) |
| **C parameter** | Inverse regularization (higher C = less regularization) |

### Decision Guide

| Metric to Optimize | When |
|-------------------|------|
| Accuracy | Balanced classes, equal error costs |
| Precision | False positives are costly (spam, fraud) |
| Recall | False negatives are costly (disease, security) |
| F1 | Balance between precision and recall |
| AUC-ROC | Overall ranking ability |

In [ ]:
# Final verification
print("=" * 60)
print("Logistic Regression Notebook Complete!")
print("=" * 60)
print("\nTopics covered:")
print("  ✅ Sigmoid function and log-odds")
print("  ✅ Binary cross-entropy loss")
print("  ✅ From-scratch implementation")
print("  ✅ Classification metrics (accuracy, precision, recall, F1)")
print("  ✅ Confusion matrices")
print("  ✅ ROC and Precision-Recall curves")
print("  ✅ Multiclass classification")
print("  ✅ Regularization (C parameter)")
print("  ✅ Coefficient interpretation (odds ratios)")